# Notebook 03 — Forecasting Models & Ensemble

**PM Accelerator | Global Weather Forecasting**

Models trained:
1. SARIMA (statsmodels)
2. Prophet (Meta)
3. XGBoost
4. LightGBM
5. LSTM (PyTorch)

Ensemble strategies:
- Simple average
- Weighted average (inverse-RMSE)
- Stacking (Ridge meta-learner)

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')

## 1. Load Data and Engineer Features

In [ ]:
from config import PROCESSED_DATA_DIR, TARGET_VARIABLE, TEST_SIZE
from feature_engineering import run_feature_engineering

daily_global = pd.read_parquet(PROCESSED_DATA_DIR / 'daily_global.parquet')
daily_global['date'] = pd.to_datetime(daily_global['date'])

feature_df = run_feature_engineering(daily_global)
print('Feature df shape:', feature_df.shape)
print('Target:', TARGET_VARIABLE)

## 2. Train All Models

In [ ]:
from forecasting_models import run_all_models

model_results = run_all_models(daily_global, feature_df)
print('\nTraining complete. Models:', list(model_results.keys()))

## 3. Metrics Summary

In [ ]:
rows = []
for name, res in model_results.items():
    row = {'Model': name}
    row.update(res.get('metrics', {}))
    rows.append(row)

metrics_df = pd.DataFrame(rows)
if 'RMSE' in metrics_df.columns:
    metrics_df = metrics_df.sort_values('RMSE')
print(metrics_df.to_string(index=False))

## 4. Forecast Visualization

In [ ]:
n_models = len(model_results)
n_display = 90  # show last 90 test points

fig, axes = plt.subplots(n_models, 1, figsize=(14, 4.5 * n_models), squeeze=False)

for ax, (model_name, res) in zip(axes.flatten(), model_results.items()):
    if 'test_preds' not in res:
        ax.set_visible(False)
        continue
    actual = np.asarray(res['test_actual'])[-n_display:]
    preds  = np.asarray(res['test_preds'])[-n_display:]
    x = np.arange(len(actual))
    ax.plot(x, actual, label='Actual', color='steelblue', lw=1.5)
    ax.plot(x, preds,  label='Predicted', color='darkorange', lw=1.5, linestyle='--')
    m = res.get('metrics', {})
    ax.set_title(f"{model_name} — MAE={m.get('MAE','N/A')} | RMSE={m.get('RMSE','N/A')}",
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Temperature (°C)')
    ax.legend(fontsize=9)

axes[-1][0].set_xlabel('Test Step')
fig.suptitle('Forecast vs Actual — Test Set (last 90 steps)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Ensemble Models

In [ ]:
from ensemble_model import run_ensemble, summarize_ensemble

ensemble_results, comparison_df = run_ensemble(model_results)
print('\nFull Model Comparison:')
print(comparison_df.to_string(index=False))

In [ ]:
if not comparison_df.empty and 'RMSE' in comparison_df.columns:
    colors = ['#2196F3' if t == 'Individual' else '#FF5722'
              for t in comparison_df.get('Type', ['Individual'] * len(comparison_df))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].barh(comparison_df['Model'], comparison_df['RMSE'], color=colors, edgecolor='white')
    axes[0].set_title('RMSE by Model', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('RMSE')
    axes[0].invert_yaxis()

    if 'MAE' in comparison_df.columns:
        axes[1].barh(comparison_df['Model'], comparison_df['MAE'], color=colors, edgecolor='white')
        axes[1].set_title('MAE by Model', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('MAE')
        axes[1].invert_yaxis()

    from matplotlib.patches import Patch
    legend_elements = [Patch(color='#2196F3', label='Individual'), Patch(color='#FF5722', label='Ensemble')]
    fig.legend(handles=legend_elements, loc='upper right')
    fig.suptitle('Individual vs Ensemble Performance', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Weighted Ensemble Detail

In [ ]:
if 'Weighted Average' in ensemble_results:
    weights = ensemble_results['Weighted Average'].get('weights', {})
    if weights:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(weights.keys(), weights.values(), color='steelblue', edgecolor='white', alpha=0.85)
        ax.set_title('Inverse-RMSE Weights per Model', fontsize=12, fontweight='bold')
        ax.set_ylabel('Weight')
        plt.tight_layout()
        plt.show()

---
**Next:** [04_spatial_analysis.ipynb](04_spatial_analysis.ipynb)